# TFMN4 PROCESSING

- ~Turn on globus endpoint:~
nspahr@poplar:/scratch1/fliu/hub_scratch/synbio/globusconnectpersonal/globusconnectpersonal-3.2.8$ nohup ./globusconnectpersonal -start &
[2] 880438
[1]   Killed                  nohup ./globusconnectpersonal -start
- Transfer into /storage/synbio/ai_synbio_data/experimental_data/downloads/
- Check file names → make file rename dictionary. Write to csv in downloads/. Create seqorder/library/received/ in seqorders/ and rename files.
- Create seqsamples.csv, measurements.csv for upload to GDrive LIMS.
- ~Start 4 fastp workers~
- Run call_mutations script (without reference genome) and inspect quality.
  ./aisynbiopipeline/pipeline/call_mutations.py  [seqordername] [no ref genome]
- Start 7 breseq workers.
- Split seqsamples into two experiments
- Run seqsamples against appropriate references
- Sort seqsample names
- Create summary and mutation comparison, Write to excel, upload.
    - For dgoA: also calculate coverage over dgoA and write out junctions around dgoA. (cant do this for verABC because missing in reference)
    - For ver experiment adapt Yusuke’s code:
        - Screen all reads for anchor seqs (4 buckets: reads with right anchor, reads with left anchor, reads with both anchors, reads with no anchor) (figure out how many errors you will allow) and cut and count barcodes (throw out partial barcodes)
        - Put into df and show relative and total abundances of verAB combos per sample as stacked bar graphs (What if too many combos bust the legend?). Samples should be grouped by lineage (11 lineages total, I think about 4 transfers selected)

## Set-Up

In [ ]:
## Make sure running in aisynbio_env

In [1]:
# Reload magic command to ensure that changes made to my imported modules are being picked up by the notebook continuously

%load_ext autoreload
%autoreload 2

In [2]:
import sys
import os

# Add project root to path for access to workflows and tasks
notebook_dir = os.getcwd()
project_root = os.path.dirname(notebook_dir)

if project_root not in sys.path:
    sys.path.insert(0, project_root)

In [3]:
import os

# Must add environment's bin to the PATH inside the notebook
env_bin = os.path.join(os.path.dirname(sys.executable), "")
os.environ["PATH"] = env_bin + ":" + os.environ["PATH"]

In [4]:
# Import LIMS utilities (use util_simple.py for standalone LIMS API usage)

%run util_simple.py

✓ LIMS API loaded successfully


## Sync Google LIMS with DB and get number of expected samples

In [ ]:
# # Run a manual LIMS mirror db sync

# from aisynbiopipeline.limsapi.sync import sync_all_sheets

# sync_all_sheets()

## Download from seq facility, file organization, renaming
- Create seqorder name and makedir into reception folder (/synbio/ai_synbio_data/experimental_data/downloads - should this be temporary?). Also make homedir to copy and view analysis reports.
- Download data into folder.
- Spot-check if duplicate read ID problem is fixed.
- Create new seqorder folder in experimental_data/sequencing_data/ and respective libraries.
- Copy all illumina fastqs into short lib, renaming in the process.

In [48]:
# Create seqorder name, make reception folder and seqorder analysis folder in nspahr homedir

from aisynbiopipeline.workflows.plasmidsaurus import create_seqorder_name
from aisynbiopipeline.workflows.seq_folder_utils import list_seqorders, SeqOrder, Library, SeqSample
import os

item_code = 'QUO1022807'
date_received = '2026-05-26'
seqorder_name = 'SeqCenter' + "_" + date_received + "_" + item_code

reception_dir = '/storage/synbio/ai_synbio_data/experimental_data/downloads/' + seqorder_name
print(reception_dir)

home_dir = '/storage/nspahr/lib_analysis/' + seqorder_name
os.makedirs(home_dir, exist_ok=True)

In [188]:
import re

def parse_Pauls_ssnames(sample_name: str) -> dict:
    """
    Parse sample names of the form:
        plate1_9_F3

    Returns:
        {
            "Plate_name": "plate1",
            "Microtiter_plate_well": "F3",
            "Transfer": 9
        }
    """

    pattern = r"^(\w+)_(\d+)_([A-Ha-h]\d{1,2})$"

    match = re.match(pattern, sample_name)

    if not match:
        raise ValueError(f"Invalid sample name format: {sample_name}")

    plate_name, transfer, well = match.groups()

    return {
        "Plate_name": plate_name,
        "Microtiter_plate_well": well.upper(),
        "Transfer": int(transfer),
    }

def parse_LIMS_ssnames(sample_name: str) -> dict:
    """
    Parse sample names of the form:
        TFMN4.exp1.fbaMUT.1.T8.P
    Returns:
        {
            "Experiment": "TFMN4",
            "Plate_name": "exp1",
            "DNA_construct": "fbaMUT"
            "Replicate": 1,
            "Transfer": 8,
            "Isolate": "P",
            "Colony_number": ''
        }
    """

    pattern = r"^(\w+).(\w+).(.*?).(\d+).T(\d{1,2})\.([PSL])([123])?$"

    match = re.match(pattern, sample_name)

    if not match:
        print(f"Unexpected sample name format: {sample_name}")
        return {
            "Experiment": None,
            "Plate_name": None,
            "DNA_construct": None,
            "Replicate": None,
            "Transfer": None,
            "Isolate": None,
            "Colony_number": None            
        }

    experiment, plate_name, construct, replicate, transfer, isolate, cnum = match.groups()

    return {
        "Experiment": experiment,
        "Plate_name": plate_name,
        "DNA_construct": construct,
        "Replicate": int(replicate),
        "Transfer": int(transfer),
        "Isolate": isolate,
        "Colony_number": cnum
    }  

In [113]:
# Get list of seqcenter samples and parse sample names to get plate/well location.

from aisynbiopipeline.workflows.fastq_utils import create_manifest

folder = '/storage/synbio/ai_synbio_data/experimental_data/downloads/SeqCenter_2026-05-26_QUO1022807/Illumina DNA Reads'
manifest = create_manifest(folder, platform='seqcenter_illumina')

for i in ["Plate_name", "Microtiter_plate_well", "Transfer"]:
    manifest.insert(len(manifest.columns), i, pd.NA)

for index, row in manifest.iterrows():
    pauls_sname_dict = parse_Pauls_ssnames(row['sample_name'])
    manifest.loc[index, "Plate_name"] = pauls_sname_dict["Plate_name"]
    manifest.loc[index, "Microtiter_plate_well"] = pauls_sname_dict["Microtiter_plate_well"]
    manifest.loc[index, "Transfer"] = pauls_sname_dict["Transfer"]

manifest['Plate_name'] = manifest['Plate_name'].apply(lambda x: x.replace('plate', 'exp'))
print(len(manifest))

246


In [77]:
# Get list of robotic samples from LIMS

rsamples = query_lims('Robotic_ALE_samples', filters = {'Experiment': "TFMN4"})
len(rsamples)

120

In [120]:
# Merge robotic samples and seqcenter samples

merged = pd.merge(rsamples, manifest, on = ["Plate_name", "Microtiter_plate_well"], how='outer')
print(len(merged))
# merged

# merged = merged[['Database_ID', 'Name', 'Experiment', 'Type', 'Condition', 'Strain_name',
#        'Transforming_DNA', 'Protocol', 'Parent_sample', 'Replicate_number',
#        'Replicate_samples', 'Robotic_run', 'Plate_name',
#        'Microtiter_plate_well', 'Plotting_group_number', 'Plotting_group_name',
#        'Transfer']]

272


In [122]:
merged

,Database_ID,Name,Experiment,Type,Condition,Strain_name,Transforming_DNA,Protocol,Parent_sample,Replicate_number,Replicate_samples,Robotic_run,Plate_name,Microtiter_plate_well,Plotting_group_number,Plotting_group_name,Select_transfers_to_sequence,deleted,last_synced,row_hash,sample_name,rvs_fastq,fwd_fastq,Transfer
0,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_30_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,30
1,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_8_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,8
2,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_9_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,9
3,,TFMN4.exp1.combo5aWT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo5aWT,,,,"TFMN4.exp1.combo5WT.1, TFMN4.exp1.combo5WT.2, ...",TFMN4,exp1,B11,10,combo5WT,"8, 9, 30",0,2026-05-15T15:41:53.663213,d985539cd86ee0c639133255b7c2047bb0ccb65c43f398...,plate1_30_B11,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,30
4,,TFMN4.exp1.combo5aWT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo5aWT,,,,"TFMN4.exp1.combo5WT.1, TFMN4.exp1.combo5WT.2, ...",TFMN4,exp1,B11,10,combo5WT,"8, 9, 30",0,2026-05-15T15:41:53.663213,d985539cd86ee0c639133255b7c2047bb0ccb65c43f398...,plate1_8_B11,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
267,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G5,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.667952,370a149ed61e129e42485d6836a423a2b4e06a92649e2b...,plate2_3_G5,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3
268,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G6,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.668033,40d693ca0e069a0949bc10e0deb585475845225289ac46...,plate2_3_G6,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3
269,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G7,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.668114,696125ac46ecbedafd51199016f01fa63eb23e4094314c...,plate2_3_G7,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3
270,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G8,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.668195,6e06608ac3029805c5a0880f4d87c1a6b8929b07d208b5...,plate2_3_G8,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3


In [143]:
# Write samples back to csv for insertion in LIMS robotic samples table

stransfers = manifest[["Plate_name", "Microtiter_plate_well", "Transfer"]].copy()
toLIMSrobotic = pd.merge(rsamples, stransfers, on = ["Plate_name", "Microtiter_plate_well"], how='outer')
groupbycols = ['Name', 'Experiment', 'Type', 'Condition', 'Strain_name',
       'Transforming_DNA', 'Protocol', 'Parent_sample', 'Replicate_number',
       'Replicate_samples', 'Robotic_run', 'Plate_name',
       'Microtiter_plate_well', 'Plotting_group_number', 'Plotting_group_name']
toLIMSrobotic = toLIMSrobotic[groupbycols + ['Transfer']]
toLIMSrobotic['Transfer'] = toLIMSrobotic['Transfer'].fillna('')
toLIMSrobotic['Transfer'] = toLIMSrobotic['Transfer'].astype(str)
toLIMSrobotic = toLIMSrobotic.groupby(groupbycols).agg(list).reset_index()
toLIMSrobotic['Transfer'] = toLIMSrobotic['Transfer'].apply(lambda x: ', '.join(x))
toLIMSrobotic.to_csv('/storage/nspahr/tmp/rsamples.csv')

In [110]:
# Merge robotic samples and condensed seqcenter samples to 

merged = pd.merge(rsamples, stransfers, on = ["Plate_name", "Microtiter_plate_well"], how='outer')
print(len(merged))
merged

merged = merged[['Database_ID', 'Name', 'Experiment', 'Type', 'Condition', 'Strain_name',
       'Transforming_DNA', 'Protocol', 'Parent_sample', 'Replicate_number',
       'Replicate_samples', 'Robotic_run', 'Plate_name',
       'Microtiter_plate_well', 'Plotting_group_number', 'Plotting_group_name',
       'Transfer']]

120


In [123]:
# From manifest merged to robotic samples, retrieve all existing seqsamples and assign name

ssamples = merged.loc[~pd.isna(merged['Transfer'])]
ssamples['Sequencing sample'] = ssamples['Name'] + ".T" + ssamples['Transfer'].astype(str) + ".P"
ssamples.head()

,Database_ID,Name,Experiment,Type,Condition,Strain_name,Transforming_DNA,Protocol,Parent_sample,Replicate_number,Replicate_samples,Robotic_run,Plate_name,Microtiter_plate_well,Plotting_group_number,Plotting_group_name,Select_transfers_to_sequence,deleted,last_synced,row_hash,sample_name,rvs_fastq,fwd_fastq,Transfer
0,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_30_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,30
1,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_8_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,8
2,,TFMN4.exp1.combo10MUT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo10MUT,,,,"TFMN4.exp1.combo10MUT.1, TFMN4.exp1.combo10MUT...",TFMN4,exp1,B10,9,combo10MUT,"8, 9, 30",0,2026-05-15T15:41:53.662128,b94e97897409a467372985598b08ac813414d03db7eeae...,plate1_9_B10,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,9
3,,TFMN4.exp1.combo5aWT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo5aWT,,,,"TFMN4.exp1.combo5WT.1, TFMN4.exp1.combo5WT.2, ...",TFMN4,exp1,B11,10,combo5WT,"8, 9, 30",0,2026-05-15T15:41:53.663213,d985539cd86ee0c639133255b7c2047bb0ccb65c43f398...,plate1_30_B11,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,30
4,,TFMN4.exp1.combo5aWT.1,TFMN4,mt_plate,Pyruvate 20 mM + Kan,ACN2586,combo5aWT,,,,"TFMN4.exp1.combo5WT.1, TFMN4.exp1.combo5WT.2, ...",TFMN4,exp1,B11,10,combo5WT,"8, 9, 30",0,2026-05-15T15:41:53.663213,d985539cd86ee0c639133255b7c2047bb0ccb65c43f398...,plate1_8_B11,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,8
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
266,,TFMN4.exp2.ACN3749.noDNA.3,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3749,,,,,,TFMN4,exp2,G4,1,ACN3749\nnoDNA,,0,2026-05-15T15:41:53.664498,f0d3c8daef5477c01b97e5e2665a7b1471da51ed80cedc...,plate2_4_G4,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,4
267,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G5,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.667952,370a149ed61e129e42485d6836a423a2b4e06a92649e2b...,plate2_3_G5,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3
268,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G6,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.668033,40d693ca0e069a0949bc10e0deb585475845225289ac46...,plate2_3_G6,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3
269,,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,TFMN4,mt_plate,Methoxybenzoate 4 mM + Kan,ACN3788,verABLib_small,,,,,TFMN4,exp2,G7,14,ACN3788\nconcY_smallLib_PCR_DpnI_cleanup,,0,2026-05-15T15:41:53.668114,696125ac46ecbedafd51199016f01fa63eb23e4094314c...,plate2_3_G7,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...,3


In [146]:
# Write existing transfers back to csv for insertion in LIMS sequencing samples table

cols = input().split('\t')

 Sequencing sample	Seqorder	Experiment	Sample Name	Sequencing plate	Sequencing plate well	Sequencing plate well row	Sequencing plate well column	Population or Single colony?	Robotic run plate	Robotic run plate well	Robotic run plate well row	Robotic run plate well column


In [147]:
cols

['Sequencing sample',
 'Seqorder',
 'Experiment',
 'Sample Name',
 'Sequencing plate',
 'Sequencing plate well',
 'Sequencing plate well row',
 'Sequencing plate well column',
 'Population or Single colony?',
 'Robotic run plate',
 'Robotic run plate well',
 'Robotic run plate well row',
 'Robotic run plate well column']

In [148]:
ssamples.columns

Index(['Database_ID', 'Name', 'Experiment', 'Type', 'Condition', 'Strain_name',
       'Transforming_DNA', 'Protocol', 'Parent_sample', 'Replicate_number',
       'Replicate_samples', 'Robotic_run', 'Plate_name',
       'Microtiter_plate_well', 'Plotting_group_number', 'Plotting_group_name',
       'Select_transfers_to_sequence', 'deleted', 'last_synced', 'row_hash',
       'sample_name', 'rvs_fastq', 'fwd_fastq', 'Transfer',
       'Sequencing sample'],
      dtype='object')

In [153]:
toLIMSseq = ssamples[['Experiment', 'Name', 'Robotic_run', 'Plate_name',
       'Microtiter_plate_well', 'Transfer', 'Sequencing sample']].copy()
toLIMSseq.rename(columns={
    'Name': 'Sample Name',
    'Transfer': 'Robotic run plate',
    'Microtiter_plate_well': 'Robotic run plate well'
}, inplace=True)
toLIMSseq['Seqorder'] = seqorder_name
toLIMSseq['Sequencing plate'] = toLIMSseq['Robotic run plate']
toLIMSseq['Sequencing plate well'] = toLIMSseq['Robotic run plate well']
toLIMSseq['Robotic run plate well row'] = toLIMSseq['Sequencing plate well'].apply(lambda x: x[0])
toLIMSseq['Robotic run plate well column'] = toLIMSseq['Sequencing plate well'].apply(lambda x: x[1:])
toLIMSseq['Sequencing plate well row'] = toLIMSseq['Robotic run plate well row']
toLIMSseq['Sequencing plate well column'] = toLIMSseq['Robotic run plate well column']
toLIMSseq['Population or Single colony?'] = toLIMSseq['Sequencing sample'].apply(lambda x: parse_LIMS_ssnames(x)['Isolate'])
toLIMSseq = toLIMSseq[cols]
toLIMSseq.to_csv('/storage/nspahr/tmp/ssamples.csv')

In [162]:
# SeqCenter provides ???.

import shutil

shutil.copy2(os.path.join(reception_dir, "Illumina DNA Reads/SeqCenter DNA Sequencing Methods.pdf"), home_dir)
shutil.copy2(os.path.join(reception_dir, "Illumina DNA Reads/DNA Sequencing Stats.xlsx"), home_dir)

'/storage/nspahr/lib_analysis/SeqCenter_2026-05-26_QUO1022807/DNA Sequencing Stats.xlsx'

In [163]:
# Create new seqorder folder in experimental_data/sequencing_data/ and libraries

seqorder = SeqOrder(seqorder_name, create=True)
short = Library(seqorder, 'Illumina', create=True)
# long = Library(seqorder, 'Nanopore', create=True)

In [166]:
ssamples.columns

Index(['Database_ID', 'Name', 'Experiment', 'Type', 'Condition', 'Strain_name',
       'Transforming_DNA', 'Protocol', 'Parent_sample', 'Replicate_number',
       'Replicate_samples', 'Robotic_run', 'Plate_name',
       'Microtiter_plate_well', 'Plotting_group_number', 'Plotting_group_name',
       'Select_transfers_to_sequence', 'deleted', 'last_synced', 'row_hash',
       'sample_name', 'rvs_fastq', 'fwd_fastq', 'Transfer',
       'Sequencing sample'],
      dtype='object')

In [ ]:
# Copy fastqs into respective library folder

from pathlib import Path
import shutil

for index, row in ssamples.iterrows():
    aisynbio_fwd_name = row['Sequencing sample'] + '_R1.fastq.gz'
    aisynbio_rvs_name = row['Sequencing sample'] + '_R2.fastq.gz'
    shutil.copy2(row['fwd_fastq'], short.path/'received'/aisynbio_fwd_name)
    shutil.copy2(row['rvs_fastq'], short.path/'received'/aisynbio_rvs_name)

In [170]:
len(os.listdir(short.path/'received'))

492

## Creating Measurements

In [172]:
measurements = ssamples.copy()
measurements['Type'] = 'Short_DNA_reads'
measurements['Protocol'] = pd.NA
measurements['Who measured'] = 'Paul Hanke & technicians'
measurements['Lab'] = 'ANL & SeqCenter'
measurements['Timestamp'] = '05-26-2026'

measurements.rename(columns={
    'Sequencing sample': 'Name',
    'Name': 'Sample ID'
}, inplace=True)

measurements = measurements[['Name', 'Type', 'Experiment', 'Sample ID', 'Protocol', 'Who measured', 'Lab', 'Timestamp']]
measurements
measurements.to_csv(f'/storage/nspahr/tmp/{seqorder_name}_measurements.csv')

## Define seqsamples

In [174]:
seqsamples = [SeqSample(short, row['Sequencing sample']) for _, row in ssamples.iterrows()]
dgoa_ss = []
ver_ss = []
for x in seqsamples:
    plate = parse_LIMS_ssnames(x.sample_name)['Plate_name']
    if plate == 'exp1':
        dgoa_ss.append(x)
    if plate == 'exp2':
        ver_ss.append(x)

# Add parents:

parentlib = Library(SeqOrder('Plasmidsaurus_2025-09-30_P4CYGL'), "Illumina")
parent_2586 = SeqSample(parentlib, 'ANLstock.ACN2586.colony3')
dgoa_ss.append(parent_2586)

parentlib = Library(SeqOrder('Plasmidsaurus_2026-05-15_PMWYSM'), "Illumina")
parent_3788 = SeqSample(parentlib, 'ANLstock.ACN3788.colony1')
ver_ss.append(parent_3788)

## Run Breseq

In [23]:
from aisynbiopipeline.workflows.reference_utils import list_reference_genomes

list_reference_genomes()

['ADP1_Neidle_CDM.gbk',
 'ACN2821_CDM.gbk',
 'ACN2586_NSS.gbk',
 'ACN3500_IRZ.gbk',
 'ACN3500_NSS.gbk',
 'ACN2821_dgoA-TWISTexmpl_NSS.gbk',
 'ACN2821_CDM_dgoAEcoli_NSS.gbk',
 'ACN2586_NSS_2.gbk',
 'ACN3749_NSS.gbk',
 'ACN3788_NSS.gbk']

In [ ]:
## To run breseq celery worksers, activate micromamba, then call worker script:
"""
cd ~/code/AISynbioPipeline
export PATH="/opt/micromamba/bin/:$PATH"
eval "$(micromamba shell hook --shell bash)"
micromamba activate
micromamba activate aisynbio_env
python -m aisynbiopipeline.tasks.breseq_task 1
"""

In [ ]:
# seven breseq workers are running!

In [178]:
# Create breseq dir

short.create_subfolder('breseq')

In [179]:
short_manifest = short.create_manifest('received')
short_manifest

,sample_name,R1,R2
0,TFMN4.exp1.combo10MUT.1.T30.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
1,TFMN4.exp1.combo10MUT.1.T8.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
2,TFMN4.exp1.combo10MUT.1.T9.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
3,TFMN4.exp1.combo10MUT.2.T30.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
4,TFMN4.exp1.combo10MUT.2.T8.P,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
...,...,...,...
241,TFMN4.exp2.ACN3788.concY_largeLib_PCR_DpnI_cle...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
242,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
243,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...
244,TFMN4.exp2.ACN3788.concY_smallLib_PCR_DpnI_cle...,/storage/synbio/ai_synbio_data/experimental_da...,/storage/synbio/ai_synbio_data/experimental_da...


In [232]:
!celery purge --queues breseq

         There is no undo for this operation!

(to skip this prompt use the -f option)
Are you sure you want to delete all tasks? [y/N]: ^C
Aborted!


In [180]:
# Where should this code go?

# Specifies and assigns breseq parameters

from pathlib import Path

def define_breseq_params(seqsample, ref_filename, poly=True, fold_coverage=300, num_processors=4, polymorphism_frequency_cutoff=0.05):

    # Normalize Path → str - Celery tasks only accept certain input data types.
    def norm(x): return str(x) if isinstance(x, Path) else x
    
    breseq_params = {
        'read_paths': [norm(x) for x in seqsample.trimmed],
        'breseq_folder': norm(seqsample.breseq),
        'reference': ref_filename,
        'polymorphism_prediction': poly,
        'limit_fold_coverage': fold_coverage,
        'num_processors': num_processors,
        'polymorphism_frequency_cutoff': polymorphism_frequency_cutoff
        
    }
    return breseq_params

In [240]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

dgoa_results = []

for sample in dgoa_ss:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN2586_NSS_2.gbk', fold_coverage=0),
        queue='breseq'
    )
    dgoa_results.append(result)    

In [244]:
########## CHECKING CELERY QUEUE ######################
## Does not include tasks that were already pre-fetched by workers!!

with client.connection_for_read() as conn:
    q = conn.SimpleQueue("breseq")
    n = q.qsize()
    q.close()

print(n)

135


In [ ]:
########## PURGING CELERY QUEUE ######################

from kombu import Connection

def purge_queue(celery_app, queue_name):
    with celery_app.connection_for_write() as conn:
        channel = conn.channel()
        count = channel.queue_purge(queue_name)
        channel.close()
    return count

removed = purge_queue(client, "breseq")
print(f"Purged {removed} tasks from breseq")

In [241]:
from celery import Celery
import os

# Create Celery client
client = Celery(
    'client',
    broker=os.getenv('CELERY_BROKER_URL', 'redis://bioseed_redis:6379/10'),
    backend=os.getenv('CELERY_RESULT_BACKEND', 'redis://bioseed_redis:6379/10')
)

# Submit tasks

ver_results = []

for sample in ver_ss:
    result = client.send_task(
        'breseq.run',
        kwargs=define_breseq_params(sample, ref_filename='ACN3788_NSS.gbk', fold_coverage=0),
        queue='breseq'
    )
    ver_results.append(result)    

In [255]:
sum([(r.status=='SUCCESS') for r in dgoa_results])

43

In [256]:
sum([(r.status=='SUCCESS') for r in ver_results])

0

dgoA seqsamples: [breseq version name]
ver seqsamples: [breseq version name]

In [185]:
dgoa_breseq_dir = os.path.join(home_dir, 'dgoA')
ver_breseq_dir = os.path.join(home_dir, 'ver')
os.makedirs(dgoa_breseq_dir, exist_ok=True)
os.makedirs(ver_breseq_dir, exist_ok=True)

In [186]:
# Symlink to library breseq folder
output_symlinks_dir = os.path.join(home_dir, 'symlink_to_library_breseq_folder')
os.makedirs(output_symlinks_dir)

path_to_folder = short.path / 'breseq'
dst = os.path.join(output_symlinks_dir, 'breseq')
os.symlink(path_to_folder, dst)

## Breseq analysis dgoA

In [ ]:
exp = 'TFMN4'

In [ ]:
from aisynbiopipeline.workflows.reference_utils import get_ref_genomes_path, genomic_region_from_features

genome = os.path.join(get_ref_genomes_path(), 'ACN2586_NSS_2.gbk')

def get_region_parameter(genbank_file, feature_first, feature_last):
    genome, start, stop = genomic_region_from_features(genbank_file, feature_first, feature_last)
    region = genome + ":" + str(start) + "-" + str(stop)
    return region


In [ ]:
from collections import Counter
from aisynbiopipeline.workflows.breseq import Breseq
from aisynbiopipeline.workflows.mapping import PileupCol
import numpy as np

def create_breseq_summary(seqsample_batch, version_name, output_path=None, regions=None, loci=None):
    
    breseq_objects = []
    
    for s in seqsample_batch:
        breseq_folder = s.library.path / 'breseq' / s.sample_name/ version_name
        b = Breseq.from_existing(breseq_folder)
        breseq_objects.append(b)
    
    rows = []
    
    for b in breseq_objects:
        
        row = {}
        
        try:
            b.count_reads()
            b.count_mutations()
            b.avg_coverage
            if regions:
                for key, value in regions.items():
                    b.get_region_average_coverage(value)
        except Exception as e:
            row.update({'seqsample': getattr(b, 'title', None)})
            row.update(parse_LIMS_ssnames(getattr(b, 'title', None)))
            row.update({'error': str(e),
                        'input_read_count': None,
                        'used_read_count': None,
                        'mapped_read_count': None,
                        'consensus_mutation_count': None,
                        'polymorphism_mutation_count': None,
                        'average_cov': None,}
                        )
            if regions:
                for key, value in regions.items():
                    row.update({key: None})
            rows.append(row)
            print(f"Error loading Breseq from {b.title}: {e}")
            continue
    
        row['seqsample'] = getattr(b, 'title', None)
        row.update(parse_LIMS_ssnames(getattr(b, 'title', None)))
        row['error'] = None
        row['input_read_count'] = getattr(b, 'input_read_count', None)
        row['used_read_count'] = getattr(b, 'used_read_count', None)
        row['mapped_read_count'] = getattr(b, 'mapped_read_count', None)
        row['consensus_mutation_count'] = getattr(b, 'consensus_mutation_count', None)
        row['polymorphism_mutation_count'] = getattr(b, 'polymorphism_mutation_count', None)
        row['average_cov'] = getattr(b, 'avg_coverage', None)
        
        if regions:
            for key, value in regions.items():
                row[key] = getattr(b, 'get_region_average_coverage', None)(value)
        
        if loci:
            for key, value in loci['mutations'].items():
                prefix = key
                basecalls = PileupCol(b.bam_path, value['locus']).basecalls
                locus_cov = len(basecalls)
                counts = Counter(basecalls)
                try:
                    alt_freq = counts[value['alt_allele']] / locus_cov
                except ZeroDivisionError:
                    alt_freq = None
                row[key+'_locus_cov'] = locus_cov
                row[key+'_alleleCounts'] = dict(counts)
                row[key+'_alt_allele_freq'] = alt_freq
        
        rows.append(row)
    
    breseq_summary = pd.DataFrame(rows)
    
    if regions:
        for key, value in regions.items():
            breseq_summary[key+'_CN'] = breseq_summary[key]/breseq_summary['average_cov']

    breseq_summary['is_parent'] = breseq_summary['seqsample'].apply(lambda x: x.startswith('ANL'))
    
    # Put rows and cols into correct order
    metadata_cols = ['seqsample', 'sample', 'strain', 'media', 'construct', 'replicate', 'transfer', 'is_parent']
    other_cols = [col for col in breseq_summary.columns.to_list() if col not in metadata_cols]
    breseq_summary = breseq_summary[metadata_cols + other_cols]
    breseq_summary.sort_values(
        ['is_parent', 'media', 'construct', 'strain', 'replicate', 'transfer'],
        ascending=[False, True, True, True, True, True],
        inplace=True
    )

    # Write breseq run summary to csv
    if output_path:
        breseq_summary.to_csv(output_path, index=False)
    
    return breseq_summary

In [ ]:
# These strains are supposed to have the 6007 bp promoter deletion, including verR.
# Therefore, defining 'ver cassette' as verB through omega KmR cassette. 

regions = {
    # 'dgoA-Star': get_region_parameter(genome2821, 'dgoA-optimized-ADP1', 'dgoA-optimized-ADP1'),
    'ver_cassette': get_region_parameter(genome3500, 'omega KmR cassette', 'verB'),
    'verAB': get_region_parameter(genome3500, 'verA', 'verB')
}

In [ ]:
regions

In [ ]:
# loci = {
#     'reference': 'ACN3500_NSS',
#     'mutations': {
#         'promoter_6kb.DEL': {
#             'locus': (950040, 955945),  # added 50 bp from start and subtracted 50 bp from end to accomodate slightly different start and end loci between samples
#             'ref_allele': '...',
#             'alt_allele': ''
#         },
#         'vanK_1-bp.DEL': {
#             'locus': (974618, 974618),
#             'ref_allele': 'G',
#             'alt_allele': ''
#         },
#         'ACIAD_RS08195<->fecI.SNP': {
#             'locus': (1405240, 1405241),  ### THIS IS WRONG!!!
#             'ref_allele': 'T',
#             'alt_allele': 'A'
#         },
#         'dsbD.Q75stop': {
#             'locus': (3465740, 3465740),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'iscR_2-bp.DEL': {
#             'locus': (1405240, 1405241),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'rpoD_3-bp.DEL': {
#             'locus': (2860637, 2860639),
#             'ref_allele': 'GAA',
#             'alt_allele': ''
#         },
#         'adeK_2-bp.DEL': {
#             'locus': (2883783, 2883784),
#             'ref_allele': 'GA',
#             'alt_allele': ''
#         },
#         'mnmA.R245H': {
#             'locus': (1230459, 1230459),
#             'ref_allele': 'C',
#             'alt_allele': 'T'
#         },
#         'ACIAD_RS01630_2-bp.DEL': {
#             'locus': (345201, 345202),
#             'ref_allele': 'AA',
#             'alt_allele': ''
#         }
#     }
# }

In [ ]:
regions_to_include = ['ver_cassette', 'verAB']
regions_sub = {key:regions[key] for key in regions_to_include if key in regions}

breseq_folder = home_dir + '/' + exp + '/' + breseq_version_name
os.makedirs(breseq_folder, exist_ok=True)

breseq_summary = create_breseq_summary(
    seqsamples,
    breseq_version_name,
    output_path=os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_summary.csv'),
    regions=regions_sub,
    # loci=loci
)

# create_html_comparison(
#     seqsample_batch,
#     breseq_version_name,
#     os.path.join(breseq_folder, f'{exp}_{breseq_version_name}_mutation_comparison.html')
# )

In [ ]:
# Create mutation comparison files

from aisynbiopipeline.workflows.breseq import compare_gdiff

breseq_objects = []
for s in seqsamples:
    breseq_folder = s.library.path / 'breseq' / s.sample_name/ breseq_version_name
    b = Breseq.from_existing(breseq_folder)
    breseq_objects.append(b)

reference = 'ACN3500_NSS.gbk'
gdiffs = [b.gd_file for b in breseq_objects]

breseq_outfolder = home_dir + '/' + exp + '/' + breseq_version_name

table_format = 'html'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
html = compare_gdiff(reference, outfile, gdiffs, format=table_format)

table_format = 'csv'
outfile = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}.{table_format}')
csv = compare_gdiff(reference, outfile, gdiffs, format=table_format)

In [ ]:
compare_df = pd.read_csv(csv)

In [ ]:
# Set display options and table formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 50)

# Need to exclude these because they will mess with pivoting on title
cols_with_diff_vals_for_same_mutation = ['new_read_count', 'new_read_count_basis', 'ref_read_count',
     'ref_read_count_basis', 'multiple_polymorphic_SNPs_in_same_codon',
     'repeat_new_copies', 'repeat_ref_copies',
    ]

# Exclude these because they are not of interest
cols_uninformative = ['clone', 'mutator_status', 'population', 'time', 'treatment', 'transl_table']

cols_to_keep = [col for col in compare_df.columns.to_list() if (col not in cols_with_diff_vals_for_same_mutation) and (col not in cols_uninformative)]

# Columns that should only have integer values. Some may have weird text which messes with conversion to nullable int type.
# Force to numeric
int_cols = [
    'aa_position',
    'codon_number',
    'codon_position',
    'gene_position',
    'insert_position',
    'position',
    'position_end',
    'position_start',
    'repeat_length',
    'size',
]
for col in int_cols:
    compare_df[col] = pd.to_numeric(compare_df[col], errors='coerce')

# Now that expected numeric cols are numeric, can convert to int if possible
compare_df = compare_df[cols_to_keep].convert_dtypes()

# Define columns that will form the Multiindex 
index = [col for col in compare_df.columns if (col!='title') and (col!='frequency')]

# Create the pivoted mutation frame, where distinct mutations are rows, samples are columns, values are frequencies
df = compare_df.pivot(index=index, columns = 'title', values = 'frequency').sort_index(level='position')
df = df.fillna(0)

# Order the comparison just like the summary
# df = df.loc[~(df > 0).all(axis=1)] # Only mutations that don't appear in all samples
df = df[breseq_summary['seqsample'].to_list()]

In [ ]:
# df_greater5 = df.loc[(df > 0.05).any(axis=1)].dropna(how="all")
# df_greater80 = df.loc[(df > 0.80).any(axis=1)].dropna(how="all")

In [ ]:
# # Write reformatted mutations to csv.
# df.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted.csv'))
# df_greater5.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater5.csv'))
# df_greater80.to_csv(os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted_greater80.csv'))

In [ ]:
# ------ Make highlight color dictionary ------
from openpyxl.styles import PatternFill
from openpyxl.utils import get_column_letter

def create_highlight_color_dict(index_col_names, breseq_summary):
    
    gray_fill = PatternFill(start_color="DDDDDD",
                            end_color="DDDDDD",
                            fill_type="solid")
    
    green_fill = PatternFill(start_color="E5FFCC",
                            end_color="E5FFCC",
                            fill_type="solid")
    
    white_fill = PatternFill(start_color="FFFFFF",
                            end_color="FFFFFF",
                            fill_type="solid")
    
    orange_fill = PatternFill(start_color="FFE5CC",
                            end_color="FFE5CC",
                            fill_type="solid")
    
    # ------ Assign column colors ------
    ## ALE transfersamples will be grouped by original sample and highlighted in alternating colors
    alt_colors = dict(
        zip(
            breseq_summary['sample'].unique(),
            len(breseq_summary['sample'].unique())*[white_fill, orange_fill]
        )
    )

    parents = breseq_summary.loc[breseq_summary['is_parent']]['seqsample'].to_list()
    # parent_sample_names = [x.replace("ANL.stock", "ANLstock") for x in parents]
    
    ALEsamples = breseq_summary.loc[~breseq_summary['is_parent']]
    ALEsample_colors = dict(zip(ALEsamples['seqsample'].to_list(), ALEsamples['sample'].apply(lambda x: alt_colors[x]).to_list()))
    
    highlight_colors = dict()
    highlight_colors.update(dict(zip(index_col_names, [gray_fill]*len(index))))
    highlight_colors.update(dict(zip(parents, [green_fill]*len(parents))))
    highlight_colors.update(ALEsample_colors)
    
    return highlight_colors


# ------ Build workbook (all samples, individual samples) ------

def write_mutation_comparison_wb(df, breseq_summary, excel_path):

    def get_parent_sample(sample):
        LIMSsamples = query_lims('Samples')[['Name', 'Parent_sample']]
        LIMSseqsamples = query_lims('Seqsamples')[['Sequencing_sample', 'Sample_Name']]
        parent_samples = pd.merge(
            LIMSseqsamples, 
            LIMSsamples, 
            left_on='Sample_Name', 
            right_on="Name", 
            how="left"
        ).drop_duplicates()
    
        parent_sample = parent_samples.loc[parent_samples['Sample_Name']==sample]['Parent_sample'].iloc[0]
        
        return parent_sample

    mutation_df = df.copy()

    def get_excel_cols_for_sample(excel_col_name_dict, seqsamples):
        excel_col_names = sorted([excel_col_name_dict[x] for x in seqsamples])
        first_col = excel_col_names[0]
        last_col = excel_col_names[-1]
        sheet_name = f"{first_col} to {last_col}"
        return sheet_name
        
    df_col_names = df.reset_index().columns
    excel_col_names = [get_column_letter(i) for i in range(1, len(df_col_names)+1)]
    excel_col_name_dict = dict(zip(
        df_col_names, excel_col_names
    ))
    
    with pd.ExcelWriter(excel_path, engine="openpyxl") as writer:
    
        ## First sheet contains all columns
        mutation_df.reset_index().to_excel(writer, sheet_name="all_samples", index=False)
    
        ## Subsequent sheets only show index cols + parent + all transfers of one sample
        ## Excel sheet names can only have 31 chars, which is much less than many of the seqsample names.
        ## Therefore, name sheets according to Excel column names (A, B, C...AA, AB, ...)
        
        for name, group in breseq_summary.groupby('sample', sort=False):
            if name=='NA':
                continue
            try:
                parent_sample = get_parent_sample(name)
                cols = [parent_sample] + group['seqsample'].to_list()
            except:
                cols = group['seqsample'].to_list()

            trimmed_df = mutation_df[cols]#.copy()
            trimmed_df = trimmed_df.loc[~(trimmed_df==0).all(axis=1)] # Discard mutations that are zero in all samples          
            sheet_name = get_excel_cols_for_sample(excel_col_name_dict, group['seqsample'].to_list())
            trimmed_df.reset_index().to_excel(writer, sheet_name=sheet_name, index=False)
    

# ------ Format workbook: format header, format columns ------ 

from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, PatternFill, Border, Side

def format_mutation_comparison_wb(excel_path, highlight_colors):
    
    wb = load_workbook(excel_path)
    
    ## Border formatting ----------
    thin = Side(style="thin")
    
    border = Border(
        left=thin,
        right=thin,
        top=thin,
        bottom=thin
    )
    for ws in wb.worksheets:
        for row in ws.iter_rows():
            for cell in row:
                cell.border = border
    
    for s in wb.sheetnames:
        ws = wb[s]
        header_row = ws[1]
        
        ## Header row formatting ----------
        header_font = Font(bold=True)
        header_alignment = Alignment(wrap_text=True)
        for cell in header_row:
            cell.font = header_font
            cell.alignment = header_alignment
            ws.freeze_panes = "A2"
        
        ## Highlight columns ----------
        col_names = {}
        for i, cell in enumerate(header_row):
            col_names[cell.value] = i
    
        for key, value in col_names.items():
            target_column_index = col_names[key]
            
            # Iterate through the rows and access the cell in the target column
            for row_cells in ws.iter_rows(min_row=1): # Start from the first row
                cell = row_cells[target_column_index]
                cell.fill = highlight_colors[key]
    
    wb.save(excel_path)


def write_and_format_mutation_comparison_excel(excel_path,
                                               mutation_comparison_df,
                                               index_col_names,
                                               breseq_summary):

    print("Assigning fill colors to columns.")
    highlight_colors = create_highlight_color_dict(index_col_names, breseq_summary)
    print("\tDone.")
    print("Writing mutation comparison to Excel workbook.")
    write_mutation_comparison_wb(mutation_comparison_df, breseq_summary, excel_path)
    print("\tDone.")
    print("Formatting Excel workbook.")
    format_mutation_comparison_wb(excel_path, highlight_colors)
    print("\tDone.")

In [ ]:
breseq_summary.head()

In [ ]:
# Write all three versions

cutoffs = {
    "": df,
    "_greater5": df.loc[(df > 0.05).any(axis=1)].dropna(how="all"),
    "_greater80": df.loc[(df > 0.80).any(axis=1)].dropna(how="all")
}

for suffix, dataframe in cutoffs.items():
    
    excel_path = os.path.join(breseq_outfolder, f'{exp}_{breseq_version_name}_mutation_comparison_reformatted{suffix}.xlsx')
    
    write_and_format_mutation_comparison_excel(excel_path,
                                           dataframe,
                                           index,
                                           breseq_summary)

## Screening for ver transformation library barcodes

In [253]:
mockSO.delete()

SeqOrder mockSOforBarcodeScreen at /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mockSOforBarcodeScreen was deleted.


In [254]:
## Testing pipeline script on mock seqorder containing one seqsample.

mockSO = SeqOrder('mockSOforBarcodeScreen', create=True)
testSS = SeqSample(Library('SeqCenter_2026-05-26_QUO1022807', 'Illumina'), 'TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P')

mockSS = testSS.copy(mockSO)

All associated files and folders of SeqSample TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P were copied to /storage/synbio/ai_synbio_data/experimental_data/sequencing_orders/mockSOforBarcodeScreen. Relative file paths were preserved.


In [ ]:
(aisynbio_env) nspahr@seed:~/code/AISynbioPipeline$ ./aisynbiopipeline/pipeline/barcode_screen.py mockSOforBarcodeScreen /storage/nspahr/tmp/verAB_barcodes.csv

In [257]:
pd.read_csv('/storage/nspahr//lib_analysis/mockSOforBarcodeScreen/mockSOforBarcodeScreen_amplicons_reads.csv')

,Sample,verA,verB
0,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
1,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
2,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
3,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
4,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
...,...,...,...
237,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
238,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
239,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
240,TFMN4.exp2.ACN3788.concX_largeLib_SpeI.5.T13.P...,83,42
